In [ ]:
import os

import numpy as np
import pandas as pd
from scipy.integrate import trapezoid

import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec
from matplotlib.ticker import MaxNLocator
import seaborn as sns

from mg_sampling import PROJECT_DIR
from mg_sampling.constants import RT
from mg_sampling.tools import read_plumed, read_plumed_fes, range_to_edges, edges_to_range, finitemin, finitemax, block_analysis, read_wsv, to_wsv, bootstrap
from mg_sampling.bias import bias_fn

LABELS = {
    'amber_tip3p': "Amber/TIP3P",
    'mMg_tip3p': "microMg/TIP3P",
    'nMg_tip3p': "nanoMg/TIP3P", 
    'mMg_opc': "microMg/OPC",
    'ecc_spce': "AmberECC/SPCE",
}

FORCEFIELDS_ARCHIVE = ['amber_tip3p', 'mMg_tip3p', 'nMg_tip3p', 'mMg_opc', 'ecc_spce']
FORCEFIELDS = ['mMg_tip3p']

os.makedirs('figures/biasfitting', exist_ok=True)
os.makedirs('figures/biasfitting/archive', exist_ok=True)

### Ramp correction

For now the bias is purely attractive, this will result in an increased binding affinity for larger and larger $λ$. This will lead to problems in phase space overlap and eventually hinder the round trips in replica exchange.

We add a correction:

$\bar{U}_\mathrm{bias}(X, λ) = U_\mathrm{bias}(X, λ) + α(λ)X$

where $X = n_\mathrm{O_P}$

With the objective

$ΔΔG_\mathrm{bind}(0 \rightarrow λ) = ΔG_\mathrm{bind}(λ) - ΔG_\mathrm{bind}(0) \approx 0$

We can express

$ΔG_\mathrm{bind}(λ) = G_\mathrm{bound}(λ) - G_\mathrm{free}(λ)$

And rewrite
$ΔΔG_\mathrm{bind}(0 \rightarrow λ) = ΔG_\mathrm{bound}(0 \rightarrow λ) - ΔG_\mathrm{free}(0 \rightarrow λ)$

Where we can compute with Zwanzwig’s formula

$ΔG_\mathrm{bound}(0 \rightarrow λ) = -RT\ln\langle\exp(-β\bar{U}_\mathrm{bias}(λ))\rangle_{0,\mathrm{bound}}$

$ΔG_\mathrm{free}(0 \rightarrow λ) = -RT\ln\langle\exp(-β\bar{U}_\mathrm{bias}(λ))\rangle_{0,\mathrm{free}} = 0$ by nullity of the bias in the free state.

Hence the objective 
$ΔG_\mathrm{bound}(0 \rightarrow λ) = -RT\ln\langle\exp(-β\bar{U}_\mathrm{bias}(λ))\rangle_{0,\mathrm{bound}} \approx 0$


In [ ]:
from bussilab.wham import wham
from mg_sampling.optimize_scalings import optimize_scalings, _get_acceptances, _get_w_n
from scipy.special import logsumexp
# Using WHAM to combine all simulations with varying bias scalings

scalings = np.array([0.6, 0.7, 0.8, 0.9, 1.0, 1.1, 1.2])

U_kNs = []
nops = []
nows = []
for scaling in scalings:
    colvar = read_plumed(PROJECT_DIR / f'kinetics/op_mMg_tip3p/{scaling:.1f}/COLVAR').iloc[::10]
    U_kNs.append(np.asarray(colvar['bias.bias']) / scaling)
    nops.append(np.asarray(colvar['cv.nb-0'] * 1.039661))
    nows.append(np.asarray(colvar['cv.nc-0'] * 1.039661))
u_n = np.concatenate(U_kNs) / RT
nop_n = np.concatenate(nops)
now_n = np.concatenate(nows)
u_nk = u_n[:, None] * scalings
W = wham(u_nk, T=1.0)
logw_n = W.logW
logZ_k = W.logZ

In [ ]:
fig, ax = plt.subplots()
X_n = nop_n - now_n + 6
λ_k = np.linspace(0, 1, 6)
for k in range(len(λ_k)):
    _logw_n = logw_n - u_n * λ_k[k]
    _logw_n -= logsumexp(_logw_n)
    ρ_X, X_edges = np.histogram(X_n, bins=100, density=True, weights=np.exp(_logw_n))
    X_range = edges_to_range(X_edges)
    ax.plot(X_range, ρ_X, color=plt.cm.cool(λ_k[k]), label=f"{λ_k[k]:.2f}")
ax.legend(title=r"$λ$")
ax.set_ylabel(r"$ρ[Δ(n_\mathrm{O_P} - n_\mathrm{O_W})]$")
ax.set_xlabel(r"$Δ(n_\mathrm{O_P} - n_\mathrm{O_W})$")

In [ ]:
fig, ax = plt.subplots()
X_n = nop_n
λ_k = np.linspace(0, 1, 6)
ΔG_k = []
for k in range(len(λ_k)):
    _logw_n = logw_n - u_n * λ_k[k]
    _logw_n -= logsumexp(_logw_n)
    _w_n = np.exp(_logw_n)
    ρ_X, X_edges = np.histogram(X_n, bins=100, density=True, weights=_w_n)
    X_range = edges_to_range(X_edges)
    ax.plot(X_range, ρ_X, color=plt.cm.cool(λ_k[k]), label=f"{λ_k[k]:.2f}")
    is_bound = (X_n > 0.0)
    def get_ΔG(is_bound, weights, axis=0):
        p_bound = np.average(is_bound, weights=weights, axis=-1)
        return -RT*np.log(p_bound/(1-p_bound))
    ΔGmean, ΔGerr = block_analysis(is_bound, _w_n, func=get_ΔG)
    bsΔGmean, bsΔGerr = bootstrap(is_bound, np.ones_like(is_bound), func=get_ΔG, weights=_w_n)
    ΔG_k.append((ΔGmean, ΔGerr, bsΔGmean, bsΔGerr))
ax.legend(title=r"$λ$")
ax.set_ylabel(r"$ρ(n_\mathrm{O_P})$")
ax.set_xlabel(r"$n_\mathrm{O_P}$")

ΔG_k, ΔGerr_k, bsΔG_k, bsΔGerr_k = np.asarray(ΔG_k).T

fig, ax = plt.subplots()
ax.errorbar(λ_k, ΔG_k, yerr=ΔGerr_k) 
ax.errorbar(λ_k, bsΔG_k, yerr=bsΔGerr_k) 
ax.set_ylabel(r"$ΔG_\mathrm{bind}(λ)$")
ax.set_xlabel(r"$λ$")

### Optimize α(λ)

Do it on a grid so that we can easily interpolate later.

In [ ]:
IS_BOUND = (nop_n[::10] > 0)
U_BOUND_N = u_n[::10][IS_BOUND]
NOP_BOUND_N = nop_n[::10][IS_BOUND]
LOGW_BOUND_N = logw_n[::10][IS_BOUND]
LOGW_BOUND_N -= logsumexp(LOGW_BOUND_N)
W_BOUND_N = np.exp(LOGW_BOUND_N)

def get_Δg_k(λ_k, α_k):
    u_k = [U_BOUND_N * λ + NOP_BOUND_N * α for λ, α in zip(λ_k, α_k)]
    Δg_k = [-logsumexp(-u, b=W_BOUND_N) for u in u_k]
    return np.array(Δg_k)

def loss(α_k):
    Δg_k = get_Δg_k(LAMBDA_K, α_k)
    out = np.square(Δg_k).sum()
    print("Loss:", f"{out:8.6f}", end='\r')
    return out

from scipy.optimize import minimize

LAMBDA_K = np.linspace(0, 1, 50)

res = minimize(loss, x0=np.zeros_like(LAMBDA_K))
print(res)
OPT_ALPHA_K = res.x

In [ ]:
fig, ax = plt.subplots()

ax.plot(LAMBDA_K, np.zeros_like(LAMBDA_K), marker='.', label='Initial', color='black')
ax.plot(LAMBDA_K, OPT_ALPHA_K, marker='.', label='Optimal', color='red')
ax.set_ylabel(r"$α(λ)$ [$kT$]")
ax.set_xlabel(r"$λ$")

fig, ax = plt.subplots()
ax.plot(LAMBDA_K, get_Δg_k(LAMBDA_K, np.zeros_like(LAMBDA_K)), color='black', label="Initial", marker='.')
ax.plot(LAMBDA_K, get_Δg_k(LAMBDA_K, OPT_ALPHA_K), color='red', label="Optimal", marker='.')
ax.set_ylabel(r"$ΔΔG_\mathrm{bind}(0 \rightarrow λ)$ [$kT$]")
ax.set_xlabel(r"$λ$")

### Optimize acceptances

In [ ]:
from scipy.interpolate import interp1d

In [ ]:
def _get_w_n(logw_n, u_n, λ, nop_n, α):
        _logw_n = logw_n - λ * u_n - α * nop_n
        _logw_n -= logsumexp(_logw_n)
        return np.exp(_logw_n)

def _get_p_acc(w1, w2):
    p_acc = 0.0
    for i in range(len(w1)):
        p_acc += 2 * np.sum(np.minimum(w1[i] * w2[i+1:], w1[i+1:] * w2[i])) + w1[i] * w2[i]
    return p_acc

def _get_acceptances(logw_n, u_n, λ_k, nop_n, α_k):
    w_k = [_get_w_n(logw_n, u_n, λ, nop_n, α) for λ, α in zip(λ_k, α_k)]
    acceptances = [_get_p_acc(w1, w2) for w1, w2 in zip(w_k[:-1], w_k[1:])]
    return np.array(acceptances)

_get_α_k = interp1d(LAMBDA_K, OPT_ALPHA_K, kind='cubic', fill_value='extrapolate')

def loss(x):
    λ_k = np.concatenate([[lmin], x, [lmax]])
    α_k = _get_α_k(λ_k)
    acceptances = _get_acceptances(logw_n, u_n, λ_k, nop_n, α_k)
    # Regularize: strictly increasing λ
    reg = -np.diff(λ_k)
    reg = np.sum(reg * (reg > 0))
    out = np.var(acceptances) + reg
    print("Loss:", f"{out:8.6f}", end='\r')
    return out

nrep = 12
guess = np.linspace(0, 1, nrep)
λ_k_opts = []
acceptances = []
Ms = [50, 150]
for M in Ms:
    # Simulate M independent systems for "guess" spacing with resampling and summing
    uM_n = []
    nopM_n = []
    for λ in guess:
        idx_m = [np.random.choice(len(u_n), size=200, p=_get_w_n(logw_n, u_n, λ, nop_n, _get_α_k(λ))) for _ in range(M)]
        uM_n.append(
            np.sum([u_n[idx] for idx in idx_m], axis=0)
        )
        nopM_n.append(
            np.sum([nop_n[idx] for idx in idx_m], axis=0)
        )
    uM_n = np.concatenate(uM_n)
    nopM_n = np.concatenate(nopM_n)
    uM_nk = uM_n[:, None] * guess + nopM_n[:, None] * _get_α_k(guess)
    logwM_n = wham(uM_nk, T=1.0).logW

    def loss(x):
        λ_k = np.concatenate([[0], x, [1]])
        α_k = _get_α_k(λ_k)
        acceptances = _get_acceptances(logwM_n, uM_n, λ_k, nopM_n, α_k)
        # Regularize: strictly increasing λ
        reg = -np.diff(λ_k)
        reg = np.sum(reg * (reg > 0))
        out = np.var(acceptances) + reg
        print("Loss:", f"{out:8.6f}", end='\r')
        return out
    
    x0 = guess[1:-1]
    res = minimize(loss, x0, tol=1e-5)
    print(res)
    λ_k_opt = np.concatenate([[0], res.x, [1]])
    acc = _get_acceptances(logwM_n, uM_n, λ_k_opt, nopM_n, _get_α_k(λ_k_opt))

    λ_k_opts.append(λ_k_opt)
    acceptances.append(acc)


In [ ]:
np.savetxt(PROJECT_DIR / 'kinetics/op_mMg_tip3p/opt_scalings_ramp.txt', λ_k_opts[-1], fmt='%.6f')
np.savetxt(PROJECT_DIR / 'kinetics/op_mMg_tip3p/opt_alpha_ramp.txt', RT * np.abs(_get_α_k(λ_k_opts[-1])), fmt='%.6f')  # in kJ/mol

In [ ]:
from matplotlib.collections import LineCollection
def colored_line(x, y, c, ax, **lc_kwargs):
    """
    Plot a line with a color specified along the line by a third value.

    It does this by creating a collection of line segments. Each line segment is
    made up of two straight lines each connecting the current (x, y) point to the
    midpoints of the lines connecting the current point with its two neighbors.
    This creates a smooth line with no gaps between the line segments.

    Parameters
    ----------
    x, y : array-like
        The horizontal and vertical coordinates of the data points.
    c : array-like
        The color values, which should be the same size as x and y.
    ax : Axes
        Axis object on which to plot the colored line.
    **lc_kwargs
        Any additional arguments to pass to matplotlib.collections.LineCollection
        constructor. This should not include the array keyword argument because
        that is set to the color argument. If provided, it will be overridden.

    Returns
    -------
    matplotlib.collections.LineCollection
        The generated line collection representing the colored line.
    """
    if "array" in lc_kwargs:
        warnings.warn('The provided "array" keyword argument will be overridden')

    # Default the capstyle to butt so that the line segments smoothly line up
    default_kwargs = {"capstyle": "butt"}
    default_kwargs.update(lc_kwargs)

    # Compute the midpoints of the line segments. Include the first and last points
    # twice so we don't need any special syntax later to handle them.
    x = np.asarray(x)
    y = np.asarray(y)
    x_midpts = np.hstack((x[0], 0.5 * (x[1:] + x[:-1]), x[-1]))
    y_midpts = np.hstack((y[0], 0.5 * (y[1:] + y[:-1]), y[-1]))

    # Determine the start, middle, and end coordinate pair of each line segment.
    # Use the reshape to add an extra dimension so each pair of points is in its
    # own list. Then concatenate them to create:
    # [
    #   [(x1_start, y1_start), (x1_mid, y1_mid), (x1_end, y1_end)],
    #   [(x2_start, y2_start), (x2_mid, y2_mid), (x2_end, y2_end)],
    #   ...
    # ]
    coord_start = np.column_stack((x_midpts[:-1], y_midpts[:-1]))[:, np.newaxis, :]
    coord_mid = np.column_stack((x, y))[:, np.newaxis, :]
    coord_end = np.column_stack((x_midpts[1:], y_midpts[1:]))[:, np.newaxis, :]
    segments = np.concatenate((coord_start, coord_mid, coord_end), axis=1)

    lc = LineCollection(segments, **default_kwargs)
    lc.set_array(c)  # set the colors of each segment

    return ax.add_collection(lc)

In [ ]:
fig, ax = plt.subplots()
import matplotlib as mpl

markers = ['.', 'o', '^', 's', 'p']

for M, λ_k_opt, acc, marker in zip(Ms, λ_k_opts, acceptances, markers):
    colored_line(np.arange(nrep), λ_k_opt, acc, ax, cmap='RdYlGn', norm=mpl.colors.Normalize(vmin=0, vmax=1))
    ax.scatter(np.arange(nrep), λ_k_opt, label=f'M = {M}', marker=marker, facecolors='none', edgecolors='black', zorder=-1)

ax.set_xlabel("Replica index")
ax.set_ylabel("Optimal scaling")
cax = plt.colorbar(plt.cm.ScalarMappable(cmap='RdYlGn'), ax=ax)
cax.ax.set_ylabel(r"$p_\mathrm{acc}(i \leftrightarrow i+1)$")
ax.legend()

In [ ]:
λ_k_opt = np.loadtxt(PROJECT_DIR / 'kinetics/op_mMg_tip3p/opt_scalings_ramp.txt')
α_opt = np.loadtxt(PROJECT_DIR / 'kinetics/op_mMg_tip3p/opt_alpha_ramp.txt') / RT

In [ ]:
fig, ax = plt.subplots()
X_n = nop_n - now_n + 6
for k, (λ, α) in enumerate(zip(λ_k_opt, α_opt)):
    _logw_n = logw_n - u_n * λ - nop_n * α
    _logw_n -= logsumexp(_logw_n)
    ρ_X, X_edges = np.histogram(X_n, bins=100, density=True, weights=np.exp(_logw_n))
    X_range = edges_to_range(X_edges)
    ax.plot(X_range, ρ_X, color=plt.cm.cool(λ), label=f"{λ:.2f}")
ax.legend(title=r"$λ$")
ax.set_ylabel(r"$ρ[Δ(n_\mathrm{O_P} - n_\mathrm{O_W})]$")
ax.set_xlabel(r"$Δ(n_\mathrm{O_P} - n_\mathrm{O_W})$")

In [ ]:
from scipy.interpolate import interp1d
_get_α_k = interp1d(LAMBDA_K, OPT_ALPHA_K, kind='cubic', fill_value='extrapolate')
print(*(f"{α:.6f}" for α in _get_α_k(np.linspace(0.6, 1.2, 7))))